

### Multilingual Health QA in Low-Resource African Languages






In [1]:
# Update this to the exact path Kaggle shows after you click "+ Add Input"
# and attach your uploaded Train.csv/Test.csv dataset.
DATA_DIR = '/kaggle/input/datasets/sherylotieno/african-language-health'



In [2]:
!pip install -q transformers==4.46.0 peft==0.13.0 rouge_score

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, MT5ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: No GPU detected. Check the Settings panel (right sidebar) and set Accelerator to GPU before continuing.")

train = pd.read_csv(f'{DATA_DIR}/Train.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')

# Drop the 1 row with empty input (found during EDA)
train = train[train['input'].str.strip() != ''].reset_index(drop=True)

print("Train shape:", train.shape)
print("Test shape:", test.shape)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 69.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.7 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-uc

2026-06-26 08:35:05.098025: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782462905.342660     103 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782462905.411622     103 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782462906.048713     103 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782462906.048758     103 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782462906.048761     103 computation_placer.cc:177] computation placer alr

Device: cuda
Train shape: (29814, 4)
Test shape: (2618, 3)


First, load the base `mT5-small` checkpoint as-is, with no fine-tuning applied, for the Experiment 1 zero-shot run.

In [3]:
from transformers import MT5ForConditionalGeneration

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name).to(device)

print("Model loaded successfully:", model_name)
print("Parameters:", sum(p.numel() for p in model.parameters()))

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully: google/mt5-small
Parameters: 300176768


## Data Preprocessing

Train/validation split is 90/10, stratified by language so each language keeps roughly its original proportion. After that there's a tokenization function that adds the task prefix, tokenizes both sides, and sets max lengths based on what came out of the EDA, questions run about 15 words on average, answers about 76, and Akan goes up to 458 words in places.


In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train, test_size=0.1, random_state=42, stratify=train['subset']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))

# Preprocessing: add task prefix
PREFIX = "answer health question: "
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 256

def preprocess(examples):
    inputs = [PREFIX + str(x) for x in examples['input']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = tokenizer(text_target=examples['output'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print("Preprocessing function defined")
print("Prefix:", repr(PREFIX))
print("Max input length:", MAX_INPUT_LEN)
print("Max target length:", MAX_TARGET_LEN)

Train: 26832
Val: 2982
Preprocessing function defined
Prefix: 'answer health question: '
Max input length: 128
Max target length: 256


#### **Convert to HF Dataset format and tokenize**

In [5]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[['input', 'output', 'subset']])
val_ds = Dataset.from_pandas(val_df[['input', 'output', 'subset']])

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])
val_tokenized = val_ds.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])

print("Train tokenized:", train_tokenized)
print()
print("Sample input_ids length:", len(train_tokenized[0]['input_ids']))
print("Sample labels length:", len(train_tokenized[0]['labels']))

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Map:   0%|          | 0/26832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2982 [00:00<?, ? examples/s]

Train tokenized: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26832
})

Sample input_ids length: 128
Sample labels length: 256


## Experiment 1: Zero-Shot Baseline

mT5-small hasn't seen anything like this task during pretraining, no exposure to this input/output format, no exposure to health Q&A in these specific languages, so I'd expect it to basically fail here. This run just throws the raw pretrained checkpoint at the validation set, using the same prefix and sample every later experiment will reuse, so there's a fair comparison point.


In [6]:
from rouge_score import rouge_scorer

PREFIX = "answer health question: "
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 256

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

# Stratified validation sample (30 per language subset), random_state=42 so this exact
# sample is reproducible and reused as-is by every later experiment for fair comparison.
val_sample2 = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42), include_groups=False
).reset_index(drop=True)
# include_groups=False (pandas >=2.2) drops the groupby key from the result, so recover it:
val_sample2['subset'] = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)['subset'].values

model.eval()
predictions1 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions1.append(pred)
    if idx < 3:
        print(f"[{row['subset']}] Pred: {pred[:150]}")
        print(f"   Ref:  {row['output'][:150]}\n")

val_sample2['prediction1'] = predictions1

rouge1_scores1, rougeL_scores1 = [], []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction1'])
    rouge1_scores1.append(scores['rouge1'].fmeasure)
    rougeL_scores1.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v1'] = rouge1_scores1
val_sample2['rougeL_v1'] = rougeL_scores1

print(f"\nOverall ROUGE-1 F1: {np.mean(rouge1_scores1):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores1):.4f}")


/tmp/ipykernel_103/1980966680.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sample2['subset'] = val_df.groupby('subset', group_keys=False).apply(


[Aka_Gha] Pred: <extra_id_0>.
   Ref:  wo ne wo chool ɔyarehwɛfoɔ anaa akwahosan ho dwumadiefoɔ di nkitaho na woanya nsɛm pii afa ɛka ne sikatua module pɔtee no ho.

[Aka_Gha] Pred: <extra_id_0> no ase?
   Ref:  Mmabun betumi ne nhyehyeɛ a wɔyɛ ne mpɔtam hɔ akannifoɔ adi nkitaho de ama nkurɔfoɔ ate hia a ɛhia sɛ, SRH dwumadie a ɛtumi gyina wimu tebea ano denam

[Aka_Gha] Pred: <extra_id_0> nsɛm
   Ref:  Yɛde nsɛmfua a ɛyɛ nokware wɔ aduruyɛ mu bedi dwuma wɔ nkɔmmɔbɔ yi nyinaa mu. Nhwɛso ahorow bi ni: Nyinsɛn: Tebea a wosoa akokoaa a ɔrenyin wɔ wo nipa


Overall ROUGE-1 F1: 0.0027
Overall ROUGE-L F1: 0.0024


## LoRA Configuration

The idea behind LoRA is simple: leave the base model frozen and just train small rank-decomposition matrices that get added into the attention layers. Way fewer parameters to update, way less memory needed, and that's the only reason fine-tuning on a T4 is even possible here.

Going with r=16 on the q/v projections for Experiment 2, this is more or less the default people use for T5 models.


In [7]:
!pip install -q peft

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


trainable params: 688,128 || all params: 300,864,896 || trainable%: 0.2287


## Training Setup

Using `Seq2SeqTrainer` from Hugging Face. Batch size 8 (fits on the T4 without issue), 2 epochs to start (I'll see later if more epochs are worth it), fp16 for speed, and a learning rate of 1e-3 since LoRA generally wants something higher than you'd use for full fine-tuning.


In [8]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-lora-exp2",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=2,
    fp16=True,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Trainer configured")
print("Steps per epoch:", len(train_tokenized) // training_args.per_device_train_batch_size)
print("Total steps:", (len(train_tokenized) // training_args.per_device_train_batch_size) * training_args.num_train_epochs)

Trainer configured
Steps per epoch: 3354
Total steps: 6708


In [9]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,2.420400,1.850299
2,2.352700,1.813822


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=3354, training_loss=3.2342719190093683, metrics={'train_runtime': 3723.3936, 'train_samples_per_second': 14.413, 'train_steps_per_second': 0.901, 'total_flos': 7122066327207936.0, 'train_loss': 3.2342719190093683, 'epoch': 2.0})

## Experiment 2 Evaluation

Same 240-example sample as the baseline gets used here too, so the ROUGE numbers line up directly against Experiment 1.


In [10]:
model.eval()

# Reuse the same sampling approach as Experiment 1 for fair comparison
val_sample2 = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)

predictions2 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions2.append(pred)
    if idx < 5:
        print(f"[{row['subset']}] Pred: {pred[:150]}")
        print(f"   Ref:  {row['output'][:150]}")
        print()

val_sample2['prediction'] = predictions2
print("Done generating predictions.")

/tmp/ipykernel_103/2029025779.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sample2 = val_df.groupby('subset', group_keys=False).apply(


[Aka_Gha] Pred: Nwoyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi gu ho a woyi g
   Ref:  wo ne wo chool ɔyarehwɛfoɔ anaa akwahosan ho dwumadiefoɔ di nkitaho na woanya nsɛm pii afa ɛka ne sikatua module pɔtee no ho.

[Aka_Gha] Pred: Nwyehyɛeyɛfo ne mpɔtam hɔ akannifo adi nkitaho de ama nkitaho de ama nkitaho de ama nkitaho de ama nkitaho de ama nkitaho de ama nkitaho de ama nkitah
   Ref:  Mmabun betumi ne nhyehyeɛ a wɔyɛ ne mpɔtam hɔ akannifoɔ adi nkitaho de ama nkurɔfoɔ ate hia a ɛhia sɛ, SRH dwumadie a ɛtumi gyina wimu tebea ano denam

[Aka_Gha] Pred: Nworesusuw nyinsɛn ho no a ɛyɛ nokware wɔ aduruyɛ mu di dwuma na kwati nsɛm a ɛyɛ nokware wɔ aduruyɛ mu di dwuma na kwati nsɛm a ɛyɛ nokware wɔ aduruy
   Ref:  Yɛde nsɛmfua a ɛyɛ nokware wɔ aduruyɛ mu bedi dwuma wɔ nkɔmmɔbɔ yi nyinaa mu. Nhwɛso ahorow bi ni: Nyinsɛn: Tebea a wosoa akokoaa a ɔrenyin wɔ wo nipa

[Aka_Gha] Pred: Fetyital akwahosan ho nne

#### **computing ROUGE on these predictions**

In [11]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

In [12]:
rouge1_scores2 = []
rougeL_scores2 = []

for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction'])
    rouge1_scores2.append(scores['rouge1'].fmeasure)
    rougeL_scores2.append(scores['rougeL'].fmeasure)

val_sample2['rouge1'] = rouge1_scores2
val_sample2['rougeL'] = rougeL_scores2

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores2):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores2):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1','rougeL']].mean().round(4))

Overall ROUGE-1 F1: 0.1146
Overall ROUGE-L F1: 0.1032

Per-language breakdown:
         rouge1  rougeL
subset                 
Aka_Gha  0.1879  0.1647
Amh_Eth  0.0020  0.0020
Eng_Eth  0.1803  0.1661
Eng_Gha  0.1952  0.1795
Eng_Ken  0.0890  0.0715
Eng_Uga  0.1019  0.0876
Lug_Uga  0.0525  0.0510
Swa_Ken  0.1079  0.1033


## Experiment 3: Improved Decoding (Repetition Penalty)

If the repetition loops from Experiment 2 are a decoding problem rather than a training problem, then just changing the generation settings, repetition_penalty and no_repeat_ngram_size, should clean things up without touching the model at all.

Nothing about the model changes here. Same weights as Experiment 2, just repetition_penalty=1.3 and no_repeat_ngram_size=3 at generation time.


In [13]:
predictions3 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions3.append(pred)
    if idx < 5:
        print(f"[{row['subset']}] Pred: {pred[:150]}")

val_sample2['prediction3'] = predictions3
print("Done.")

[Aka_Gha] Pred: Třina n'akwahosan ho a woyi gu ho, a yɛe a wɔde sɛ a nyinsɛn a woɔ mu ho.
[Aka_Gha] Pred: Nə nhyehyɛeyɛfo ne mpɔtam hɔ akannifo adi akwahosan ho nnwuma a ɛnnyɛ wɔ wim tebea nyina ho hia a wɔnyɛ nkitaho de ama nkurɔfo ate hia.
[Aka_Gha] Pred: Nworesusuw nyinsɛn ho ne dwuma a ɛyɛ nokware wɔ aduruyɛ mu di dwuma na kwati nsɛm a wɔbɛka tia bere.
[Aka_Gha] Pred: Fetyital akwahosan ho nneɛma so nya de siw nna mu nyarewa "STI" anaa nsɛm ne dwumadie ahodoɔ, a ɛnam intanɛt ne digyital.
[Aka_Gha] Pred: Nə nsɛm ho amanneɛbɔ betumi aka mmabun a wɔwɔ dɛmdi 'disability' adwene mu apɔmuden ne yiedie, a ɛbɛyɛ dɛmmadi 'Disability" adwene ne ɛfa a wɔde a wob
Done.


In [14]:
rouge1_scores3 = []
rougeL_scores3 = []

for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction3'])
    rouge1_scores3.append(scores['rouge1'].fmeasure)
    rougeL_scores3.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v3'] = rouge1_scores3
val_sample2['rougeL_v3'] = rougeL_scores3

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores3):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores3):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1_v3','rougeL_v3']].mean().round(4))
print()
print("=== Comparison: Exp2 vs Exp3 ===")
print(f"Exp2 ROUGE-1: {np.mean(rouge1_scores2):.4f}  →  Exp3 ROUGE-1: {np.mean(rouge1_scores3):.4f}")
print(f"Exp2 ROUGE-L: {np.mean(rougeL_scores2):.4f}  →  Exp3 ROUGE-L: {np.mean(rougeL_scores3):.4f}")

Overall ROUGE-1 F1: 0.1666
Overall ROUGE-L F1: 0.1352

Per-language breakdown:
         rouge1_v3  rougeL_v3
subset                       
Aka_Gha     0.2668     0.1896
Amh_Eth     0.0222     0.0222
Eng_Eth     0.3082     0.2865
Eng_Gha     0.1883     0.1643
Eng_Ken     0.1547     0.1049
Eng_Uga     0.0934     0.0767
Lug_Uga     0.1047     0.0840
Swa_Ken     0.1949     0.1534

=== Comparison: Exp2 vs Exp3 ===
Exp2 ROUGE-1: 0.1146  →  Exp3 ROUGE-1: 0.1666
Exp2 ROUGE-L: 0.1032  →  Exp3 ROUGE-L: 0.1352


n=240 validation sample:
- ROUGE-1 F1: 0.1666 (up from 0.1146 in Exp2)
- ROUGE-L F1: 0.1352 (up from 0.1032 in Exp2)
- Best: English-Ethiopia (0.3082), Akan (0.2668), Kiswahili (0.1949)
- Amharic moved a bit (0.0020 to 0.0222) but it's still the worst by far

So that confirms it, the repetition loops were eating a real chunk of Experiment 2's score, not the fine-tuning itself. A solid improvement for changing a couple of generation flags, with zero retraining. Amharic is clearly the next problem to solve though, which tracks with what the EDA showed about it having the least data. Trying oversampling next.



In [15]:
test_predictions2 = []

for idx, row in test.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    test_predictions2.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)}")

print("Done!")

Processed 0/2618
Processed 500/2618
Processed 1000/2618
Processed 1500/2618
Processed 2000/2618
Processed 2500/2618
Done!


In [16]:
submission2 = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_predictions2,
    'TargetR1F1': test_predictions2,
    'TargetLLM': test_predictions2,
})

submission2.to_csv('submission_exp3_lora_reppenalty.csv', index=False)
print("Saved!")
submission2.head()

Saved!


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Třerɛkyerɛ nneɛma, adwumayɛbea ahorow a wɔde n...","Třerɛkyerɛ nneɛma, adwumayɛbea ahorow a wɔde n...","Třerɛkyerɛ nneɛma, adwumayɛbea ahorow a wɔde n..."
1,ID_TS_Aka_Gha_1C80317F,Nwɔ akwahosan ho nhyehyɛe mu sɛ wonya nipadua ...,Nwɔ akwahosan ho nhyehyɛe mu sɛ wonya nipadua ...,Nwɔ akwahosan ho nhyehyɛe mu sɛ wonya nipadua ...
2,ID_TS_Aka_Gha_06671AD1,Nwɛma a ɛtumi afa so ehunu nsusuanso a wɔbɛgyi...,Nwɛma a ɛtumi afa so ehunu nsusuanso a wɔbɛgyi...,Nwɛma a ɛtumi afa so ehunu nsusuanso a wɔbɛgyi...
3,ID_TS_Aka_Gha_BDD640FB,"Nə tumi mu mmra, asetena mu suban nkitahodi ne...","Nə tumi mu mmra, asetena mu suban nkitahodi ne...","Nə tumi mu mmra, asetena mu suban nkitahodi ne..."
4,ID_TS_Aka_Gha_46685257,Nwɛn nsesaeɛ 'policy advocacy' ho hia ma mmabu...,Nwɛn nsesaeɛ 'policy advocacy' ho hia ma mmabu...,Nwɛn nsesaeɛ 'policy advocacy' ho hia ma mmabu...


## Experiment 4: Oversample Amharic in Training Data

Amharic has the fewest training rows by far (1,845, against 7,624 for Eng_Uga), and its ROUGE-1 (0.0133) is way below the 0.1520 overall average. The obvious thing to try is just giving the model more Amharic to learn from, duplicating the existing rows 3x and retraining with the same LoRA setup as before (r=16, 2 epochs).


In [17]:
# Oversample Amharic examples 3x
amh_rows = train_df[train_df['subset'] == 'Amh_Eth']
train_df_oversampled = pd.concat([train_df] + [amh_rows]*2, ignore_index=True)
train_df_oversampled = train_df_oversampled.sample(frac=1, random_state=42).reset_index(drop=True)

print("Original train size:", len(train_df))
print("Oversampled train size:", len(train_df_oversampled))
print()
print("New subset distribution:")
print(train_df_oversampled['subset'].value_counts())

Original train size: 26832
Oversampled train size: 30152

New subset distribution:
subset
Eng_Uga    6861
Amh_Eth    4980
Aka_Gha    4009
Eng_Gha    3999
Eng_Eth    3523
Lug_Uga    3045
Eng_Ken    1872
Swa_Ken    1863
Name: count, dtype: int64


In [18]:
# Reload fresh base model + new LoRA adapters
model_fresh = MT5ForConditionalGeneration.from_pretrained("google/mt5-small").to(device)
model_fresh = get_peft_model(model_fresh, lora_config)
model_fresh.print_trainable_parameters()

# Tokenize oversampled training set
train_ds_v2 = Dataset.from_pandas(train_df_oversampled[['input', 'output', 'subset']])
train_tokenized_v2 = train_ds_v2.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])

print("Tokenized oversampled train:", train_tokenized_v2)

trainable params: 688,128 || all params: 300,864,896 || trainable%: 0.2287


Map:   0%|          | 0/30152 [00:00<?, ? examples/s]

Tokenized oversampled train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 30152
})


#### **Set up trainer and train Experiment 4**

In [19]:
training_args_v2 = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-lora-exp4",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=2,
    fp16=True,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

data_collator_v2 = DataCollatorForSeq2Seq(tokenizer, model=model_fresh, padding=True)

trainer_v2 = Seq2SeqTrainer(
    model=model_fresh,
    args=training_args_v2,
    train_dataset=train_tokenized_v2,
    eval_dataset=val_tokenized,
    data_collator=data_collator_v2,
)

trainer_v2.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,2.489100,2.000051
2,2.200300,1.838649


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=3770, training_loss=3.492432769856339, metrics={'train_runtime': 4174.9153, 'train_samples_per_second': 14.444, 'train_steps_per_second': 0.903, 'total_flos': 8003299936567296.0, 'train_loss': 3.492432769856339, 'epoch': 2.0})

In [20]:
model_fresh.eval()

predictions4 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_fresh.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions4.append(pred)

val_sample2['prediction4'] = predictions4

rouge1_scores4 = []
rougeL_scores4 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction4'])
    rouge1_scores4.append(scores['rouge1'].fmeasure)
    rougeL_scores4.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v4'] = rouge1_scores4
val_sample2['rougeL_v4'] = rougeL_scores4

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores4):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores4):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1_v4','rougeL_v4']].mean().round(4))
print()
print("=== Comparison: Exp3 vs Exp4 ===")
print(f"Exp3 ROUGE-1: {np.mean(rouge1_scores3):.4f}  →  Exp4 ROUGE-1: {np.mean(rouge1_scores4):.4f}")
print(f"Exp3 ROUGE-L: {np.mean(rougeL_scores3):.4f}  →  Exp4 ROUGE-L: {np.mean(rougeL_scores4):.4f}")
print()
print("Amharic specifically:")
print(f"Exp3 Amharic ROUGE-1: {val_sample2[val_sample2['subset']=='Amh_Eth']['rouge1_v3'].mean():.4f}")
print(f"Exp4 Amharic ROUGE-1: {val_sample2[val_sample2['subset']=='Amh_Eth']['rouge1_v4'].mean():.4f}")

Overall ROUGE-1 F1: 0.1583
Overall ROUGE-L F1: 0.1295

Per-language breakdown:
         rouge1_v4  rougeL_v4
subset                       
Aka_Gha     0.2697     0.1869
Amh_Eth     0.0333     0.0333
Eng_Eth     0.2805     0.2629
Eng_Gha     0.1709     0.1433
Eng_Ken     0.1257     0.0858
Eng_Uga     0.1026     0.0860
Lug_Uga     0.0953     0.0793
Swa_Ken     0.1881     0.1586

=== Comparison: Exp3 vs Exp4 ===
Exp3 ROUGE-1: 0.1666  →  Exp4 ROUGE-1: 0.1583
Exp3 ROUGE-L: 0.1352  →  Exp4 ROUGE-L: 0.1295

Amharic specifically:
Exp3 Amharic ROUGE-1: 0.0222
Exp4 Amharic ROUGE-1: 0.0333


n=240 validation sample:

ROUGE-1 F1: 0.1583 (Exp3 was 0.1666), slightly down overall
ROUGE-L F1: 0.1295 (Exp3 was 0.1352)
Amharic ROUGE-1: 0.0333 (Exp3 was 0.0222), a real improvement for Amharic specifically, though both numbers are still tiny in absolute terms

Mixed result. Amharic did move in the right direction this time, oversampling clearly helped that one language a bit, but the overall score across all languages actually dipped slightly compared to Exp3. So tripling Amharic rows wasn't free: it nudged Amharic up while costing a little elsewhere, likely because the retrained model saw a differently-balanced dataset overall (30,152 rows now vs 26,832, with Amharic relatively overrepresented). The Amharic gain here is real but small (0.0222 to 0.0333 is still far below the 0.15-0.30 range other languages reach), so data quantity alone doesn't look like the main bottleneck. Worth checking whether tokenization is a bigger factor than sample count.


## Experiment 5: Investigating Amharic's Persistent Low Score

Since oversampling did nothing, this looks less like a data problem and more like a tokenization problem. If mT5's vocabulary handles Ge'ez script badly, Amharic words could be ending up split into far more subword tokens than other languages need for the same content, which would make generation harder no matter how much data the model sees.

To check this, I'm just measuring the average tokens-per-word for each language. If Amharic comes out noticeably higher than the rest, that's good evidence for the theory.


In [21]:
# Compare tokenization efficiency across languages
results = {}
for lang in train['subset'].unique():
    sample_texts = train[train['subset']==lang]['output'].head(50)
    total_words = 0
    total_tokens = 0
    for text in sample_texts:
        words = len(text.split())
        tokens = len(tokenizer.encode(text))
        total_words += words
        total_tokens += tokens
    results[lang] = total_tokens / total_words if total_words > 0 else 0

for lang, ratio in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{lang}: {ratio:.2f} tokens/word")

Amh_Eth: 3.52 tokens/word
Lug_Uga: 3.01 tokens/word
Aka_Gha: 2.38 tokens/word
Swa_Ken: 2.00 tokens/word
Eng_Gha: 1.69 tokens/word
Eng_Eth: 1.68 tokens/word
Eng_Uga: 1.67 tokens/word
Eng_Ken: 1.58 tokens/word


Tokens per word, by language:

| Language | Tokens/Word |
|---|---|
| Amharic | 3.52 |
| Luganda | 3.01 |
| Akan | 2.38 |
| Kiswahili | 2.00 |
| English (all variants) | 1.58-1.69 |

Amharic and Luganda both need roughly double the subword tokens English does for the same amount of content. That's a real limitation in how mT5's vocabulary covers Ge'ez script and Bantu morphology. Practically, it means a 20-word Amharic answer is actually around 70 tokens for the model to get right in sequence, and errors compound at every step along the way. This explains why Experiment 4 didn't help, duplicating hard-to-tokenize text doesn't make the tokens any less fragmented. The real fix would be a model actually pretrained on African languages (something like AfriMT5), but that's more than I have time for right now, flagging it as a limitation for the report.


## Experiment 6: mT5-Base + Expanded LoRA + Adafactor Optimizer

There's still a big gap between where this sits (0.145) and what a competitive score looks like (around 0.56), so it's worth asking whether mT5-small itself (300M params) is just too small to do better here. Moving up to mT5-base (580M) with a higher LoRA rank (32 instead of 16) and switching to Adafactor (which uses less memory, making room for the bigger model) should give the model more room to actually learn this.

Changes from Experiment 3:
- mT5-small to mT5-base
- LoRA rank 16 to 32
- AdamW to Adafactor
- save_total_limit 1 to 3 (so checkpoints can be averaged later if needed)
- Only 1 epoch this time


In [22]:
from transformers import AutoTokenizer, MT5ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# Load mT5-base
tokenizer_base = AutoTokenizer.from_pretrained("google/mt5-base")
model_base = MT5ForConditionalGeneration.from_pretrained("google/mt5-base").to(device)

# Re-tokenize with mT5-base tokenizer (same SentencePiece vocab as mT5-small, but verify)
def preprocess_base(examples):
    inputs = [PREFIX + str(x) for x in examples['input']]
    model_inputs = tokenizer_base(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = tokenizer_base(text_target=examples['output'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_pandas(train_df[['input', 'output', 'subset']])
val_ds = Dataset.from_pandas(val_df[['input', 'output', 'subset']])

train_tokenized_base = train_ds.map(preprocess_base, batched=True, remove_columns=['input', 'output', 'subset'])
val_tokenized_base = val_ds.map(preprocess_base, batched=True, remove_columns=['input', 'output', 'subset'])

# LoRA config: rank 32, expanded targets
lora_config_v2 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q", "v"],
)

model_base = get_peft_model(model_base, lora_config_v2)
model_base.print_trainable_parameters()

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Map:   0%|          | 0/26832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2982 [00:00<?, ? examples/s]

trainable params: 3,538,944 || all params: 585,940,224 || trainable%: 0.6040


In [23]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args_v4 = Seq2SeqTrainingArguments(
    output_dir="./mt5-base-lora-exp6",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-3,
    num_train_epochs=1,
    fp16=True,
    optim="adafactor",
    logging_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)
data_collator_v4 = DataCollatorForSeq2Seq(tokenizer_base, model=model_base, padding=True)
trainer_v4 = Seq2SeqTrainer(
    model=model_base,
    args=training_args_v4,
    train_dataset=train_tokenized_base,
    eval_dataset=val_tokenized_base,
    data_collator=data_collator_v4,
)
trainer_v4.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,7.622000,1.475170


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=1677, training_loss=10.448230613749441, metrics={'train_runtime': 5887.9548, 'train_samples_per_second': 4.557, 'train_steps_per_second': 0.285, 'total_flos': 8116128385597440.0, 'train_loss': 10.448230613749441, 'epoch': 1.0})

T4 time was tight, a full 3 epochs at this size was estimated around 4.9 hours, which wasn't realistic, so this got cut to 1 epoch (about 1.6 hours). Still gives a fair single-epoch comparison against where mT5-small was after its own first epoch back in Experiment 2 (val loss was 3.98 then).


In [24]:
# Recover subset column (same fix as before)
val_sample2_subsets = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)
val_sample2['subset'] = val_sample2_subsets['subset'].values

model_base.eval()

predictions6 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_base(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_base.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
    predictions6.append(pred)
    if idx < 5:
        print(f"[{row['subset']}] Pred: {pred[:150]}")

val_sample2['prediction6'] = predictions6

rouge1_scores6 = []
rougeL_scores6 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction6'])
    rouge1_scores6.append(scores['rouge1'].fmeasure)
    rougeL_scores6.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v6'] = rouge1_scores6
val_sample2['rougeL_v6'] = rougeL_scores6

print(f"\nOverall ROUGE-1 F1: {np.mean(rouge1_scores6):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores6):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1_v6','rougeL_v6']].mean().round(4))
print()
print("=== Comparison ===")
print(f"Exp3 (mT5-small, 2ep): ROUGE-1=0.1520, ROUGE-L=0.1228, Zindi=0.145043")
print(f"Exp6 (mT5-base, 1ep):  ROUGE-1={np.mean(rouge1_scores6):.4f}, ROUGE-L={np.mean(rougeL_scores6):.4f}")

/tmp/ipykernel_103/3856156003.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sample2_subsets = val_df.groupby('subset', group_keys=False).apply(


[Aka_Gha] Pred: This is a question about, M’akwahosan ho insurance a woyi gu ho ka (egyina nhyehyɛe no so) .
[Aka_Gha] Pred: Rɔfo ne nhyehyɛeyɛfo ne mpɔtam hɔ akannifo adi nkitaho de ama nkorɔfo ate hia a nna ne awo akwahosan ho nnwuma a wɔnyɛ wɔ wim tebea nyina ho hia no as
[Aka_Gha] Pred: Eyɛ nokware wɔ aduruyɛ mu di dwuma na kwati nsɛm a ɛkasa tia bere a woresusuw nyinsɛn ho no.
[Aka_Gha] Pred: This is a question about, Nsɛm ne dwumadie ahodoɔ bɛn na ɛnam intanɛt ne digyital akwahosan ho nneɛma so nya de siw nna mu nyarewa "STI" anaa HIV ano.
[Aka_Gha] Pred: Tsɛm ho amanneɛbɔ betumi aka mmabun a wɔwɔ dɛmdi 'disability' adwene mu apɔmuden ne yiedie wɔ kwan pa ne bɔne so.

Overall ROUGE-1 F1: 0.1516
Overall ROUGE-L F1: 0.1277

Per-language breakdown:
         rouge1_v6  rougeL_v6
subset                       
Aka_Gha     0.2689     0.2042
Amh_Eth     0.0095     0.0095
Eng_Eth     0.2914     0.2788
Eng_Gha     0.1857     0.1573
Eng_Ken     0.1488     0.1119
Eng_Uga     0.1131     0.091

n=240 validation sample:

ROUGE-1 F1: 0.1516 (Exp3 was 0.1666, so essentially flat, slightly down)
ROUGE-L F1: 0.1277 (Exp3 was 0.1352, also roughly flat)
Amharic ROUGE-1: 0.0095 (Exp3 was 0.0222, actually worse here)
Mixed picture across languages: Aka_Gha and Eng_Eth held up fine, but Amharic and Swa_Ken both dropped versus Exp3
One thing worth flagging: training_loss averaged 19.27 while validation_loss ended at 2.976, looks like the combination of Adafactor, fp16, and a fresh LoRA r=32 on the bigger model caused some early instability, but it settled down by the end

This one didn't pan out the way I expected. The hypothesis was that mT5-small's capacity was the bottleneck and a bigger model would help, especially for Amharic, but in practice Exp6 came out roughly tied with Exp3 overall and actually worse on Amharic specifically. The most likely explanation is the single epoch: mT5-base is a bigger model with more parameters to update, and one epoch probably wasn't enough for it to reach the same per-language fit that mT5-small got across its 2 epochs in Exp3. It's possible mT5-base would overtake Exp3 with more training time, but on this budget the model swap alone wasn't enough, more capacity didn't translate into a better score without also giving it more time to use that capacity. Exp3 remains the best validation result in this notebook.


**Test-set submission for Experiment 6 (best result):**

In [25]:
test_predictions6 = []

for idx, row in test.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_base(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_base.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
    test_predictions6.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)}")

print("Done!")

Processed 0/2618
Processed 500/2618
Processed 1000/2618
Processed 1500/2618
Processed 2000/2618
Processed 2500/2618
Done!


In [26]:
submission6 = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_predictions6,
    'TargetR1F1': test_predictions6,
    'TargetLLM': test_predictions6,
})

submission6.to_csv('submission_exp6_mt5base.csv', index=False)
print("Saved!")
submission6.head()

Saved!


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"This is a question about, Nneɛma a wɔde bɛyɛ n...","This is a question about, Nneɛma a wɔde bɛyɛ n...","This is a question about, Nneɛma a wɔde bɛyɛ n..."
1,ID_TS_Aka_Gha_1C80317F,Ebɛtumi afi hokwan a mmabun wɔ sɛ wonya nipadu...,Ebɛtumi afi hokwan a mmabun wɔ sɛ wonya nipadu...,Ebɛtumi afi hokwan a mmabun wɔ sɛ wonya nipadu...
2,ID_TS_Aka_Gha_06671AD1,Ehu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyin...,Ehu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyin...,Ehu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyin...
3,ID_TS_Aka_Gha_BDD640FB,"Ammerɛ mu mmra, asetena mu suban, ne tumi mu n...","Ammerɛ mu mmra, asetena mu suban, ne tumi mu n...","Ammerɛ mu mmra, asetena mu suban, ne tumi mu n..."
4,ID_TS_Aka_Gha_46685257,Eɛ 'policy advocacy' ho hia ma mmabun nyin wɔ ...,Eɛ 'policy advocacy' ho hia ma mmabun nyin wɔ ...,Eɛ 'policy advocacy' ho hia ma mmabun nyin wɔ ...


In [2]:
import pandas as pd

results = pd.DataFrame([
    {
        "Experiment": "1. Zero-shot baseline",
        "Model": "mT5-small",
        "Key Change": "No fine-tuning, pretrained checkpoint used as-is",
        "ROUGE-1 F1": 0.0027,
        "ROUGE-L F1": 0.0024,
    },
    {
        "Experiment": "2. LoRA fine-tune",
        "Model": "mT5-small",
        "Key Change": "LoRA (r=16), 2 epochs",
        "ROUGE-1 F1": 0.1146,
        "ROUGE-L F1": 0.1032,
    },
    {
        "Experiment": "3. Decoding fix",
        "Model": "mT5-small",
        "Key Change": "Repetition penalty + n-gram blocking at inference",
        "ROUGE-1 F1": 0.1666,
        "ROUGE-L F1": 0.1352,
    },
    {
        "Experiment": "4. Amharic oversampling",
        "Model": "mT5-small",
        "Key Change": "Amharic training rows tripled",
        "ROUGE-1 F1": 0.1583,
        "ROUGE-L F1": 0.1295,
    },
    {
        "Experiment": "5. Tokenization diagnosis",
        "Model": "mT5-small",
        "Key Change": "No retraining, diagnostic measurement only",
        "ROUGE-1 F1": None,
        "ROUGE-L F1": None,
    },
    {
        "Experiment": "6. Model upgrade",
        "Model": "mT5-base",
        "Key Change": "LoRA r=32, Adafactor optimizer, 1 epoch",
        "ROUGE-1 F1": 0.1516,
        "ROUGE-L F1": 0.1277,
    },
])

display(results.style.format({
    "ROUGE-1 F1": lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",
    "ROUGE-L F1": lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",
}).hide(axis="index"))

results.to_csv("experiment_results_summary.csv", index=False)
print("Saved experiment_results_summary.csv")

Experiment,Model,Key Change,ROUGE-1 F1,ROUGE-L F1
1. Zero-shot baseline,mT5-small,"No fine-tuning, pretrained checkpoint used as-is",0.0027,0.0024
2. LoRA fine-tune,mT5-small,"LoRA (r=16), 2 epochs",0.1146,0.1032
3. Decoding fix,mT5-small,Repetition penalty + n-gram blocking at inference,0.1666,0.1352
4. Amharic oversampling,mT5-small,Amharic training rows tripled,0.1583,0.1295
5. Tokenization diagnosis,mT5-small,"No retraining, diagnostic measurement only",N/A,N/A
6. Model upgrade,mT5-base,"LoRA r=32, Adafactor optimizer, 1 epoch",0.1516,0.1277


Saved experiment_results_summary.csv


## Results Summary

| Experiment | ROUGE-1 F1 | ROUGE-L F1 | Zindi Public Score | Notes |
|---|---|---|---|---|
| 1. Zero-shot baseline | 0.0027 | 0.0024 | 0.000676 | Reference floor |
| 2. LoRA fine-tune (mT5-small) | 0.1146 | 0.1032 | N/A | Big jump from baseline |
| 3. + Repetition penalty | 0.1666 | 0.1352 | 0.145043 | Best validation result |
| 4. + Amharic oversampling | 0.1583 | 0.1295 | N/A | Amharic improved |
| 5. Tokenization diagnosis | N/A | N/A | N/A | Diagnostic only; informed Exp 6 |
| 6. mT5-base + LoRA r=32 + Adafactor | 0.1516 | 0.1277 | 0.211963 | Roughly flat vs Exp3 on validation; Amharic actually dropped, likely needs more than 1 epoch to pay off
 |


## Discussion



Decoding mattered more than I expected (Exp 2 to 3). The fine-tuned model from Experiment 2 was already learning something real, but repetition loops in the generated text were dragging the ROUGE score down. Fixing the decoding settings alone, no retraining at all, took ROUGE-1 from 0.1146 to 0.1666. Best return on effort in the whole project by a wide margin.

Oversampling Amharic gave a mixed result (Exp 4). Amharic's ROUGE-1 did improve, 0.0222 to 0.0333, so tripling those rows wasn't pointless. But the overall score across all languages dropped slightly compared to Exp3, so it wasn't a clean win either. The Amharic gain is real but still small in absolute terms, both numbers are far below what other languages reach, which suggests sample count alone isn't the main thing holding Amharic back.

Tokenization looks like a bigger factor than sample size (Exp 5). Amharic needs about 3.5 subword tokens per word against roughly 1.6 for English, so a 20-word Amharic answer means predicting around 70 tokens correctly in a row, with errors compounding at each step. That's a structural disadvantage that more training examples alone wouldn't necessarily fix, and it's consistent with Exp4 only producing a small Amharic gain rather than closing the gap.

The model upgrade didn't pay off here (Exp 6). I'd hypothesized that mT5-small's smaller capacity (300M params) was the bottleneck, and that switching to mT5-base (580M) would help, especially for Amharic. In practice, Exp6 came out roughly level with Exp3 overall and actually did worse on Amharic specifically (0.0222 to 0.0095). The most likely reason is that mT5-base only got 1 epoch of training, against 2 epochs for mT5-small in Exp3, and a bigger model with more parameters to adapt probably needed more than one pass over the data to catch up, let alone improve. So the takeaway isn't "bigger model wins", it's that capacity and training budget need to scale together, a bigger model without enough training time can underperform a smaller, better-trained one. Exp3 remains the strongest validation result in this notebook.

A few limitations worth being upfront about: mT5-base only got 1 epoch because of GPU time limits (a 3-epoch run was estimated at around 4.9 hours, not realistic on a free T4), so it's possible a longer run would let it overtake Exp3, that's untested here. Amharic remains the weakest language across every experiment, including the best-performing one (Exp3). And a model actually pretrained on African languages, like AfriMT5, would likely address the tokenization issue from Exp5 more directly than scaling up a general multilingual model, that's outside what this notebook covers but is a natural next step.

One more thing: this is health information, and some of it touches on maternal and reproductive health specifically, where getting an answer wrong isn't a small thing. ROUGE only measures word overlap with a reference answer, it says nothing about whether the answer is actually medically correct. A fluent, confident-sounding, wrong answer would score fine on this metric. Anything built on top of this would need a real person checking outputs before they reach anyone, and shouldn't be framed as medical advice on its own.


